In [1]:
# %% [markdown]
# # CHRONOSPHERE: Automated Insight Extraction
# ## Discovering Patterns Through AI-Powered Analysis

# %% [markdown]
# ### 1. Setup

# %%
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from intelligence.insight_generator import InsightGenerator
from core.data_processor import AnomalyDetector
import warnings
warnings.filterwarnings('ignore')

print("✅ Modules loaded")

# %%
# Load data
print("Loading data...")
df = pd.read_csv('../data/processed/events_cleaned.csv', parse_dates=['event_time'])
hourly = pd.read_csv('../data/processed/hourly_aggregated.csv', parse_dates=['event_time'])
daily = pd.read_csv('../data/processed/daily_aggregated.csv', parse_dates=['date'])

print(f"✅ Data loaded")


✅ Modules loaded
Loading data...
✅ Data loaded


In [2]:
# %% [markdown]
# ### 2. Generate All Insights

# %%
print("Generating automated insights...")
print("=" * 70)

# Initialize insight generator
generator = InsightGenerator(daily, 'date')

# Generate insights for key metrics
metric_columns = ['unique_users', 'total_events', 'unique_products', 'total_revenue']
insights = generator.generate_all_insights(metric_columns)

print(f"\n✅ Generated {len(insights)} insights")

Generating automated insights...

✅ Generated 15 insights


In [3]:
# %% [markdown]
# ### 3. Display All Insights

# %%
# Generate and print full report
report = generator.generate_summary_report()
print(report)

# %%
# Save report to file
with open('../reports/INSIGHTS_REPORT.txt', 'w') as f:
    f.write(report)
print("\n✅ Report saved to ../reports/INSIGHTS_REPORT.txt")

CHRONOSPHERE INSIGHT REPORT


CORRELATION INSIGHTS
----------------------------------------------------------------------
1. 🔗 total_events and total_revenue show very strong positive correlation (r=0.99). When total_events increases, total_revenue tends to increase as well.

2. 🔗 unique_users and unique_products show very strong positive correlation (r=0.98). When unique_users increases, unique_products tends to increase as well.

3. 🔗 total_events and unique_products show very strong positive correlation (r=0.97). When total_events increases, unique_products tends to increase as well.

4. 🔗 unique_products and total_revenue show very strong positive correlation (r=0.96). When unique_products increases, total_revenue tends to increase as well.

5. 🔗 unique_users and total_events show very strong positive correlation (r=0.95). When unique_users increases, total_events tends to increase as well.

6. 🔗 unique_users and total_revenue show very strong positive correlation (r=0.96). When un

In [4]:
# %% [markdown]
# ### 4. Top 5 Most Important Insights

# %%
print("\n" + "=" * 70)
print("TOP 5 MOST IMPORTANT INSIGHTS")
print("=" * 70 + "\n")

top_insights = generator.get_top_insights(n=5)

for i, insight in enumerate(top_insights, 1):
    print(f"{i}. {insight['text']}")
    print(f"   [Importance: {insight.get('importance', 0)}/100]")
    print()



TOP 5 MOST IMPORTANT INSIGHTS

1. 🔗 total_events and total_revenue show very strong positive correlation (r=0.99). When total_events increases, total_revenue tends to increase as well.
   [Importance: 79/100]

2. 🔗 unique_users and unique_products show very strong positive correlation (r=0.98). When unique_users increases, unique_products tends to increase as well.
   [Importance: 78/100]

3. 🔗 total_events and unique_products show very strong positive correlation (r=0.97). When total_events increases, unique_products tends to increase as well.
   [Importance: 77/100]

4. 🔗 unique_products and total_revenue show very strong positive correlation (r=0.96). When unique_products increases, total_revenue tends to increase as well.
   [Importance: 77/100]

5. 🔗 unique_users and total_events show very strong positive correlation (r=0.95). When unique_users increases, total_events tends to increase as well.
   [Importance: 76/100]



In [5]:
# %% [markdown]
# ### 5. Detailed Anomaly Analysis

# %%
print("=" * 70)
print("DETAILED ANOMALY ANALYSIS")
print("=" * 70)

# Detect anomalies with multiple methods
detector = AnomalyDetector(daily, 'date')

# Method 1: Z-score
daily_zscore = detector.detect_zscore('total_events', threshold=2.5)
anomalies_zscore = daily_zscore[daily_zscore['total_events_anomaly']].copy()

print(f"\n1. Z-Score Method (threshold=2.5):")
print(f"   Detected {len(anomalies_zscore)} anomalies")
print("\n   Anomaly dates and values:")
for idx, row in anomalies_zscore.iterrows():
    print(f"   • {row['date'].strftime('%Y-%m-%d')}: {row['total_events']:,.0f} events (z-score: {row['total_events_zscore']:.2f})")

# %%
# Method 2: IQR
daily_iqr = detector.detect_iqr('total_events', multiplier=1.5)
anomalies_iqr = daily_iqr[daily_iqr['total_events_anomaly_iqr']].copy()

print(f"\n2. IQR Method (multiplier=1.5):")
print(f"   Detected {len(anomalies_iqr)} anomalies")

# %%
# Get context for top anomaly
if len(anomalies_zscore) > 0:
    top_anomaly_idx = anomalies_zscore['total_events'].idxmax()
    
    print(f"\n3. Deep Dive: Top Anomaly")
    print(f"   Date: {anomalies_zscore.loc[top_anomaly_idx, 'date'].strftime('%Y-%m-%d')}")
    print(f"   Events: {anomalies_zscore.loc[top_anomaly_idx, 'total_events']:,.0f}")
    print(f"   Z-score: {anomalies_zscore.loc[top_anomaly_idx, 'total_events_zscore']:.2f}")
    
    # Get 7-day context
    context = detector.get_anomaly_context(top_anomaly_idx, window_hours=7*24)
    
    print(f"\n   7-Day Context Window:")
    print(context[['date', 'total_events', 'unique_users', 'total_revenue']].to_string(index=False))

DETAILED ANOMALY ANALYSIS
Detected 1 anomalies in total_events (z-score > 2.5)

1. Z-Score Method (threshold=2.5):
   Detected 1 anomalies

   Anomaly dates and values:
   • 2019-10-02: 188,101 events (z-score: 3.17)
Detected 3 anomalies in total_events (IQR method)

2. IQR Method (multiplier=1.5):
   Detected 3 anomalies

3. Deep Dive: Top Anomaly
   Date: 2019-10-02
   Events: 188,101
   Z-score: 3.17

   7-Day Context Window:
      date  total_events  unique_users  total_revenue
2019-10-01        132534         18794      990959.44
2019-10-02        188101         33310     1375565.75
2019-10-03        116247         15871      868097.06
2019-10-04        108172         14353      770710.29
2019-10-05         99011         14614      731120.45
2019-10-06        173299         30960     1242370.06
2019-10-07        168631         25947     1233772.32
2019-10-08        137515         19686      987076.74
2019-10-09        129953         17702      945231.46


In [6]:
# %% [markdown]
# ### 6. Correlation Analysis

# %%
print("\n" + "=" * 70)
print("CORRELATION ANALYSIS")
print("=" * 70)

# Compute correlation matrix
corr_cols = ['unique_users', 'total_events', 'unique_products', 'total_revenue', 'unique_sessions']
corr_matrix = daily[corr_cols].corr()

print("\nCorrelation Matrix:")
print(corr_matrix.round(3))

# %%
# Find strongest correlations
print("\nStrongest Correlations:")
correlations = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        corr_val = corr_matrix.iloc[i, j]
        correlations.append((col1, col2, corr_val))

correlations.sort(key=lambda x: abs(x[2]), reverse=True)

for col1, col2, corr_val in correlations[:5]:
    direction = "positive" if corr_val > 0 else "negative"
    strength = "very strong" if abs(corr_val) > 0.8 else "strong" if abs(corr_val) > 0.6 else "moderate"
    print(f"• {col1} ↔ {col2}: {corr_val:.3f} ({strength} {direction})")


CORRELATION ANALYSIS

Correlation Matrix:
                 unique_users  total_events  unique_products  total_revenue  \
unique_users            1.000         0.954            0.979          0.958   
total_events            0.954         1.000            0.970          0.993   
unique_products         0.979         0.970            1.000          0.965   
total_revenue           0.958         0.993            0.965          1.000   
unique_sessions         0.960         0.985            0.965          0.979   

                 unique_sessions  
unique_users               0.960  
total_events               0.985  
unique_products            0.965  
total_revenue              0.979  
unique_sessions            1.000  

Strongest Correlations:
• total_events ↔ total_revenue: 0.993 (very strong positive)
• total_events ↔ unique_sessions: 0.985 (very strong positive)
• unique_users ↔ unique_products: 0.979 (very strong positive)
• total_revenue ↔ unique_sessions: 0.979 (very strong positi

In [7]:
# %% [markdown]
# ### 7. Temporal Pattern Analysis

# %%
print("\n" + "=" * 70)
print("TEMPORAL PATTERN ANALYSIS")
print("=" * 70)

# Weekly patterns
if 'day_of_week' in df.columns:
    weekly_pattern = df.groupby('day_of_week')['event_type'].count()
    peak_day = weekly_pattern.idxmax()
    trough_day = weekly_pattern.idxmin()
    
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    print(f"\n1. Weekly Patterns:")
    print(f"   Peak day: {day_names[peak_day]} ({weekly_pattern[peak_day]:,} events)")
    print(f"   Trough day: {day_names[trough_day]} ({weekly_pattern[trough_day]:,} events)")
    print(f"   Peak/Trough ratio: {weekly_pattern[peak_day] / weekly_pattern[trough_day]:.2f}x")

# %%
# Hourly patterns
if 'hour' in df.columns:
    hourly_pattern = df.groupby('hour')['event_type'].count()
    peak_hour = hourly_pattern.idxmax()
    trough_hour = hourly_pattern.idxmin()
    
    print(f"\n2. Daily (Hourly) Patterns:")
    print(f"   Peak hour: {peak_hour}:00 ({hourly_pattern[peak_hour]:,} events)")
    print(f"   Trough hour: {trough_hour}:00 ({hourly_pattern[trough_hour]:,} events)")
    print(f"   Peak/Trough ratio: {hourly_pattern[peak_hour] / hourly_pattern[trough_hour]:.2f}x")

# %%
# Weekend vs Weekday
if 'is_weekend' in df.columns:
    weekend_events = df[df['is_weekend'] == 1].shape[0]
    weekday_events = df[df['is_weekend'] == 0].shape[0]
    
    weekend_avg = weekend_events / df[df['is_weekend'] == 1]['date'].nunique()
    weekday_avg = weekday_events / df[df['is_weekend'] == 0]['date'].nunique()
    
    print(f"\n3. Weekend vs Weekday:")
    print(f"   Weekend daily average: {weekend_avg:,.0f} events")
    print(f"   Weekday daily average: {weekday_avg:,.0f} events")
    
    if weekday_avg > 0:
        diff_pct = ((weekend_avg - weekday_avg) / weekday_avg) * 100
        direction = "higher" if diff_pct > 0 else "lower"
        print(f"   Weekend is {abs(diff_pct):.1f}% {direction} than weekday")


TEMPORAL PATTERN ANALYSIS

1. Weekly Patterns:
   Peak day: Wednesday (687,158 events)
   Trough day: Saturday (400,758 events)
   Peak/Trough ratio: 1.71x

2. Daily (Hourly) Patterns:
   Peak hour: 19:00 (246,055 events)
   Trough hour: 1:00 (36,933 events)
   Peak/Trough ratio: 6.66x

3. Weekend vs Weekday:
   Weekend daily average: 113,421 events
   Weekday daily average: 126,705 events
   Weekend is 10.5% lower than weekday


In [8]:
# %% [markdown]
# ### 8. User Behavior Insights

# %%
print("\n" + "=" * 70)
print("USER BEHAVIOR INSIGHTS")
print("=" * 70)

# User activity distribution
user_activity = df.groupby('user_id').size()

print(f"\n1. User Activity Distribution:")
print(f"   Total users: {len(user_activity):,}")
print(f"   Mean events per user: {user_activity.mean():.2f}")
print(f"   Median events per user: {user_activity.median():.0f}")
print(f"   Std dev: {user_activity.std():.2f}")

# %%
# Power users analysis
percentiles = [50, 75, 90, 95, 99]
print(f"\n2. User Activity Percentiles:")
for p in percentiles:
    threshold = user_activity.quantile(p/100)
    print(f"   {p}th percentile: {threshold:.0f} events")

# Top 1% power users
top_1_pct_threshold = user_activity.quantile(0.99)
power_users = user_activity[user_activity > top_1_pct_threshold]
power_user_events = power_users.sum()
total_events = len(df)

print(f"\n3. Power Users (Top 1%):")
print(f"   Number of power users: {len(power_users):,}")
print(f"   Events by power users: {power_user_events:,}")
print(f"   Percentage of all events: {(power_user_events/total_events)*100:.1f}%")
print(f"   Average events per power user: {power_users.mean():.0f}")

# %%
# Session analysis
session_events = df.groupby('user_session').size()

print(f"\n4. Session Analysis:")
print(f"   Total sessions: {len(session_events):,}")
print(f"   Mean events per session: {session_events.mean():.2f}")
print(f"   Median events per session: {session_events.median():.0f}")

# Single-event sessions (bounce rate proxy)
single_event_sessions = (session_events == 1).sum()
bounce_rate = (single_event_sessions / len(session_events)) * 100
print(f"   Single-event sessions: {single_event_sessions:,} ({bounce_rate:.1f}%)")


USER BEHAVIOR INSIGHTS

1. User Activity Distribution:
   Total users: 389,483
   Mean events per user: 9.81
   Median events per user: 2
   Std dev: 35.62

2. User Activity Percentiles:
   50th percentile: 2 events
   75th percentile: 4 events
   90th percentile: 18 events
   95th percentile: 43 events
   99th percentile: 145 events

3. Power Users (Top 1%):
   Number of power users: 3,875
   Events by power users: 1,046,498
   Percentage of all events: 27.4%
   Average events per power user: 270

4. Session Analysis:
   Total sessions: 851,378
   Mean events per session: 4.49
   Median events per session: 1
   Single-event sessions: 506,086 (59.4%)


In [9]:
# %% [markdown]
# ### 9. Conversion Funnel Deep Dive

# %%
print("\n" + "=" * 70)
print("CONVERSION FUNNEL ANALYSIS")
print("=" * 70)

# Overall funnel
total_views = (df['event_type'] == 'view').sum()
total_cart = (df['event_type'] == 'cart').sum()
total_purchase = (df['event_type'] == 'purchase').sum()

view_to_cart = (total_cart / total_views) * 100 if total_views > 0 else 0
cart_to_purchase = (total_purchase / total_cart) * 100 if total_cart > 0 else 0
view_to_purchase = (total_purchase / total_views) * 100 if total_views > 0 else 0

print(f"\n1. Overall Conversion Funnel:")
print(f"   Views: {total_views:,}")
print(f"   Cart additions: {total_cart:,}")
print(f"   Purchases: {total_purchase:,}")
print(f"\n   View → Cart: {view_to_cart:.2f}%")
print(f"   Cart → Purchase: {cart_to_purchase:.2f}%")
print(f"   View → Purchase: {view_to_purchase:.2f}%")

# %%
# Funnel by time of day
if 'time_of_day' in df.columns:
    print(f"\n2. Conversion by Time of Day:")
    
    for tod in ['Morning', 'Afternoon', 'Evening', 'Night']:
        tod_data = df[df['time_of_day'] == tod]
        tod_views = (tod_data['event_type'] == 'view').sum()
        tod_purchases = (tod_data['event_type'] == 'purchase').sum()
        tod_conv = (tod_purchases / tod_views) * 100 if tod_views > 0 else 0
        
        print(f"   {tod}: {tod_conv:.2f}%")

# %%
# Funnel by day of week
if 'is_weekend' in df.columns:
    print(f"\n3. Conversion by Day Type:")
    
    # Weekend
    weekend_data = df[df['is_weekend'] == 1]
    weekend_views = (weekend_data['event_type'] == 'view').sum()
    weekend_purchases = (weekend_data['event_type'] == 'purchase').sum()
    weekend_conv = (weekend_purchases / weekend_views) * 100 if weekend_views > 0 else 0
    
    # Weekday
    weekday_data = df[df['is_weekend'] == 0]
    weekday_views = (weekday_data['event_type'] == 'view').sum()
    weekday_purchases = (weekday_data['event_type'] == 'purchase').sum()
    weekday_conv = (weekday_purchases / weekday_views) * 100 if weekday_views > 0 else 0
    
    print(f"   Weekend: {weekend_conv:.2f}%")
    print(f"   Weekday: {weekday_conv:.2f}%")
    
    if weekday_conv > 0:
        diff = ((weekend_conv - weekday_conv) / weekday_conv) * 100
        direction = "higher" if diff > 0 else "lower"
        print(f"   Weekend is {abs(diff):.1f}% {direction} than weekday")


CONVERSION FUNNEL ANALYSIS

1. Overall Conversion Funnel:
   Views: 1,815,733
   Cart additions: 1,190,259
   Purchases: 242,842

   View → Cart: 65.55%
   Cart → Purchase: 20.40%
   View → Purchase: 13.37%

2. Conversion by Time of Day:
   Morning: 13.90%
   Afternoon: 13.44%
   Evening: 12.61%
   Night: 13.34%

3. Conversion by Day Type:
   Weekend: 12.15%
   Weekday: 13.76%
   Weekend is 11.7% lower than weekday


In [10]:
# %% [markdown]
# ### 10. Price Analysis

# %%
print("\n" + "=" * 70)
print("PRICE ANALYSIS")
print("=" * 70)

# Overall price statistics
purchase_data = df[df['event_type'] == 'purchase']

print(f"\n1. Purchase Price Statistics:")
print(f"   Mean: ${purchase_data['price'].mean():.2f}")
print(f"   Median: ${purchase_data['price'].median():.2f}")
print(f"   Std dev: ${purchase_data['price'].std():.2f}")
print(f"   Min: ${purchase_data['price'].min():.2f}")
print(f"   Max: ${purchase_data['price'].max():.2f}")

# %%
# Price categories
if 'price_category' in df.columns:
    print(f"\n2. Purchases by Price Category:")
    price_cat_purchases = purchase_data['price_category'].value_counts()
    
    for cat in ['Budget', 'Low', 'Medium', 'High', 'Premium']:
        if cat in price_cat_purchases.index:
            count = price_cat_purchases[cat]
            pct = (count / len(purchase_data)) * 100
            print(f"   {cat}: {count:,} ({pct:.1f}%)")

# %%
# Revenue analysis
total_revenue = purchase_data['price'].sum()
daily_revenue = daily['total_revenue'].mean()

print(f"\n3. Revenue Metrics:")
print(f"   Total revenue: ${total_revenue:,.2f}")
print(f"   Average daily revenue: ${daily_revenue:,.2f}")
print(f"   Average order value: ${purchase_data['price'].mean():.2f}")

# %% [markdown]
# ### 11. Key Findings Summary

# %%
print("\n" + "=" * 70)
print("🔍 KEY FINDINGS SUMMARY")
print("=" * 70)

key_findings = f"""
1. TEMPORAL PATTERNS:
   • Peak activity: Hour {hourly_pattern.idxmax()}:00
   • Most active day: {day_names[weekly_pattern.idxmax()]}
   • Weekend traffic is {abs(diff_pct):.1f}% {'higher' if diff_pct > 0 else 'lower'} than weekday

2. USER BEHAVIOR:
   • Top 1% of users generate {(power_user_events/total_events)*100:.1f}% of all events
   • Median user engagement: {user_activity.median():.0f} events
   • Session bounce rate: {bounce_rate:.1f}%

3. CONVERSION METRICS:
   • Overall conversion rate: {view_to_purchase:.2f}%
   • Cart abandonment rate: {100 - cart_to_purchase:.1f}%
   • Weekend conversion is {'higher' if weekend_conv > weekday_conv else 'lower'} than weekday

4. ANOMALIES DETECTED:
   • {len(anomalies_zscore)} significant anomalies identified
   • Largest spike: {anomalies_zscore['total_events'].max():,.0f} events

5. REVENUE INSIGHTS:
   • Average order value: ${purchase_data['price'].mean():.2f}
   • Total revenue: ${total_revenue:,.2f}
   • Most popular price range: {price_cat_purchases.index[0]}

6. CORRELATIONS:
   • Strongest correlation: {correlations[0][0]} ↔ {correlations[0][1]} (r={correlations[0][2]:.3f})
   • User activity strongly predicts revenue
"""

print(key_findings)

# %%
# Save key findings
with open('../reports/KEY_FINDINGS.txt', 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("CHRONOSPHERE PROJECT - KEY FINDINGS\n")
    f.write("=" * 70 + "\n\n")
    f.write(key_findings)
    f.write("\n" + "=" * 70 + "\n")

print("\n✅ Key findings saved to ../reports/KEY_FINDINGS.txt")


PRICE ANALYSIS

1. Purchase Price Statistics:
   Mean: $4.78
   Median: $3.00
   Std dev: $6.71
   Min: $0.32
   Max: $98.41

2. Purchases by Price Category:
   Budget: 219,211 (90.3%)
   Low: 22,598 (9.3%)
   Medium: 1,033 (0.4%)

3. Revenue Metrics:
   Total revenue: $1,161,094.70
   Average daily revenue: $898,381.85
   Average order value: $4.78

🔍 KEY FINDINGS SUMMARY

1. TEMPORAL PATTERNS:
   • Peak activity: Hour 19:00
   • Most active day: Wednesday
   • Weekend traffic is 10.5% lower than weekday

2. USER BEHAVIOR:
   • Top 1% of users generate 27.4% of all events
   • Median user engagement: 2 events
   • Session bounce rate: 59.4%

3. CONVERSION METRICS:
   • Overall conversion rate: 13.37%
   • Cart abandonment rate: 79.6%
   • Weekend conversion is lower than weekday

4. ANOMALIES DETECTED:
   • 1 significant anomalies identified
   • Largest spike: 188,101 events

5. REVENUE INSIGHTS:
   • Average order value: $4.78
   • Total revenue: $1,161,094.70
   • Most popular pri

In [11]:
# %% [markdown]
# ### 12. Actionable Recommendations

# %%
print("\n" + "=" * 70)
print("💡 ACTIONABLE RECOMMENDATIONS")
print("=" * 70)

recommendations = f"""
Based on the analysis, here are data-driven recommendations:

1. OPTIMIZE PEAK HOURS:
   • Focus marketing efforts around hour {hourly_pattern.idxmax()}:00
   • Ensure adequate server capacity during peak times
   • Schedule promotions to align with high-traffic periods

2. REDUCE CART ABANDONMENT:
   • Current cart abandonment: {100 - cart_to_purchase:.1f}%
   • Implement cart recovery emails
   • Simplify checkout process
   • Add urgency indicators (limited stock, time-limited offers)

3. ENGAGE POWER USERS:
   • Top 1% drives {(power_user_events/total_events)*100:.1f}% of activity
   • Create VIP program for high-engagement users
   • Personalized recommendations for power users
   • Early access to new products

4. WEEKEND STRATEGY:
   • Weekend conversion is {'higher' if weekend_conv > weekday_conv else 'lower'} than weekday
   • {'Increase' if weekend_conv > weekday_conv else 'Adjust'} weekend marketing budget
   • Tailor weekend promotions accordingly

5. INVESTIGATE ANOMALIES:
   • {len(anomalies_zscore)} spike days detected
   • Analyze what drove these spikes
   • Replicate successful campaigns
   • Prepare for predictable peaks (holidays, events)

6. PRICE OPTIMIZATION:
   • Most purchases in {price_cat_purchases.index[0]} range
   • Test price points in adjacent categories
   • Bundle products to increase average order value
   • Current AOV: ${purchase_data['price'].mean():.2f}
"""

print(recommendations)

# %%
# Save recommendations
with open('../reports/RECOMMENDATIONS.txt', 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("CHRONOSPHERE PROJECT - ACTIONABLE RECOMMENDATIONS\n")
    f.write("=" * 70 + "\n\n")
    f.write(recommendations)
    f.write("\n" + "=" * 70 + "\n")

print("\n✅ Recommendations saved to ../reports/RECOMMENDATIONS.txt")


💡 ACTIONABLE RECOMMENDATIONS

Based on the analysis, here are data-driven recommendations:

1. OPTIMIZE PEAK HOURS:
   • Focus marketing efforts around hour 19:00
   • Ensure adequate server capacity during peak times
   • Schedule promotions to align with high-traffic periods

2. REDUCE CART ABANDONMENT:
   • Current cart abandonment: 79.6%
   • Implement cart recovery emails
   • Simplify checkout process
   • Add urgency indicators (limited stock, time-limited offers)

3. ENGAGE POWER USERS:
   • Top 1% drives 27.4% of activity
   • Create VIP program for high-engagement users
   • Personalized recommendations for power users
   • Early access to new products

4. WEEKEND STRATEGY:
   • Weekend conversion is lower than weekday
   • Adjust weekend marketing budget
   • Tailor weekend promotions accordingly

5. INVESTIGATE ANOMALIES:
   • 1 spike days detected
   • Analyze what drove these spikes
   • Replicate successful campaigns
   • Prepare for predictable peaks (holidays, events)

In [12]:
# %% [markdown]
# ### 13. Final Summary

# %%
final_summary = f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    CHRONOSPHERE ANALYSIS COMPLETE                    ║
╚══════════════════════════════════════════════════════════════════════╝

📊 INSIGHTS GENERATED: {len(insights)}
🎯 TOP INSIGHTS: {len(top_insights)}
⚠️  ANOMALIES DETECTED: {len(anomalies_zscore)}
📈 CONVERSION RATE: {view_to_purchase:.2f}%
💰 TOTAL REVENUE: ${total_revenue:,.2f}
👥 UNIQUE USERS: {df['user_id'].nunique():,}

✅ All analyses complete
✅ Reports saved to /reports/
✅ Ready for presentation

╚══════════════════════════════════════════════════════════════════════╝
"""

print(final_summary)

# %%
print("✅ Insight extraction and analysis complete!")
print("📄 All reports generated")
print("🚀 Project ready for final presentation!")


╔══════════════════════════════════════════════════════════════════════╗
║                    CHRONOSPHERE ANALYSIS COMPLETE                    ║
╚══════════════════════════════════════════════════════════════════════╝

📊 INSIGHTS GENERATED: 15
🎯 TOP INSIGHTS: 5
⚠️  ANOMALIES DETECTED: 1
📈 CONVERSION RATE: 13.37%
💰 TOTAL REVENUE: $1,161,094.70
👥 UNIQUE USERS: 389,483

✅ All analyses complete
✅ Reports saved to /reports/
✅ Ready for presentation

╚══════════════════════════════════════════════════════════════════════╝

✅ Insight extraction and analysis complete!
📄 All reports generated
🚀 Project ready for final presentation!
